In [ ]:
# customers:
# customer_id, first_name, last_name, birthdate, email, phone, country, created_at

# properties:
# property_id, name, city, capacity, stars

# room_types:
# room_type_id, type, max_capacity, base_cost

# rooms:
# room_id, room_number, property_id, room_type_id

# promotions:
# promotion_id, promotion_code, discount_pct, campaign, valid_from, valid_to

# bookings:
# booking_id, customer_id, room_id, number_of_nights, check_in_date, check_out_date, status, is_full_package, promotion_id, total_amount

# payments:
# payment_id, booking_id, amount, bank_provider, status, created_at

# departments:
# department_id, name

# employees:
# employee_id, first_name, last_name, birthdate, hire_date, property_id, manager_id, department_id

In [16]:
# import libraries
import pandas as pd
import numpy as np
from faker import Faker # to generate realistic names cities etc ...
from pathlib import Path # to create output folder that holds generated csv's


# define constants
SEED = 201  # for repudecing same random generation evreytime

# numpy default range
rng = np.random.default_rng(SEED)

# defining Faker seed
Faker.seed(SEED)
fake = Faker(["en_US", "fr_FR", "de_DE", "es_ES", "pt_PT", "pl_PL", "es_MX"])
fake_us = Faker(["en_US"]) # for phone number format and cities names

ROOM_TYPES = {"Single": (1, 75),  "Double": (2, 125), "Triple": (3, 210), "Suite": (4, 495)}

DEPARTMENTS = ["Management", "Front Desk", "Housekeeping", "Maintenance",
               "Food & Beverage", "Sales & Marketing", "Finance"]

CAMPAIGNS = ["Black Friday", "Thanksgiving", "New Year", "Weekly", "Monthly"]

#-------------------------------------------------------------------------------- HELPERS

def make_ids(name: str, n_rows: int) -> list[str]:
    return [f"{name}_{i:05d}" for i in range(1, n_rows+1)]

def generate_dates(start, end, n_rows: int) -> pd.DatetimeIndex:

    s = np.broadcast_to(np.asarray(pd.to_datetime(start), dtype= "datetime64[D]"), (n_rows, ))
    e = np.broadcast_to(np.asarray(pd.to_datetime(end), dtype= "datetime64[D]"), (n_rows, ))
    span = np.maximum((e - s).astype(int), 0)
    offsets = (rng.random(n_rows) * (span + 1)).astype(int)

    return pd.to_datetime(s + offsets.astype("timedelta64[D]"))


def adult_from(birthdates: pd.DatetimeIndex) -> pd.DatetimeIndex:
    return birthdates + pd.DateOffset(years= 18)



#------------------------------------------------------------------------------------ Tables
def generate_customers(n_rows: int) -> pd.DataFrame:

    birthdate = generate_dates("1950-01-01", "2003-01-01", n_rows)
    earliest = np.maximum(adult_from(birthdate), pd.Timestamp("2018-01-01"))
    
    return pd.DataFrame({
    "customer_id" : make_ids("cust", n_rows),
    "first_name" : [fake.first_name() for _ in range(n_rows)], 
    "last_name" : [fake.last_name() for _ in range(n_rows)], 
    "birthdate": birthdate,
    "email" : [fake.unique.email() for _ in range(n_rows)], 
    "phone" : [fake_us.unique.phone_number() for _ in range(n_rows)],
    "country" : [fake.country() for _ in range(n_rows)],
    "created_at" : generate_dates(earliest, pd.Timestamp.today().normalize(), n_rows)
    })



def generate_room_types() -> pd.DataFrame:

    n_rows = len(list(ROOM_TYPES))
    types = [list(ROOM_TYPES)[i] for i in range(n_rows)]
    cap = [ROOM_TYPES.get(i)[0] for i in types]
    cost = [ROOM_TYPES.get(i)[1] * rng.uniform(0.9, 1.2) for i in types]
    
    return pd.DataFrame({
        "room_type_id": make_ids("rtype", n_rows), 
        "type": types, 
        "max_capacity": cap, 
        "base_cost": np.round(cost, 2)
    })
    


def generate_rooms(properties_ids: list[str], room_types_ids: list[str], n_rows: int) -> pd.DataFrame:
    
    df = pd.DataFrame({
    "room_id": make_ids("room", n_rows), 

    "property_id": rng.choice(properties_ids, size= n_rows), 
    "room_type_id": rng.choice(room_types_ids, size= n_rows)
    })

    df["room_number"] = df.groupby("property_id").cumcount() + 1

    return df[["room_id", "room_number", "property_id", "room_type_id"]]



def generate_proprties(rooms: pd.DataFrame, room_types: pd.DataFrame, ids, n_rows: int) -> pd.DataFrame:

    cap = rooms.merge(room_types[["room_type_id", "max_capacity"]], on="room_type_id").groupby("property_id")["max_capacity"].sum()

    return pd.DataFrame({
        "property_id": ids, 
        "name": [f"{fake.unique.last_name()} {fake.unique.word()}" for _ in range(n_rows)], 
        "city": [fake_us.unique.city() for _ in range(n_rows)], 
        "capacity": pd.Series(ids).map(cap).fillna(0).astype(int), 
        "stars": rng.integers(3, 6, size= n_rows)
    })




def generate_promotions(n_rows: int) -> pd.DataFrame:

    valid_from = generate_dates("2022-10-12", "2026-08-01", n_rows)
    valid_to = valid_from + pd.DateOffset(days= rng.integers(2, 15))
    
    return pd.DataFrame({
        "promotion_id": make_ids("promo", n_rows), 
        "promotion_code": [fake_us.unique.word().title() for _ in range(n_rows)], 
        "discount_pct": rng.integers(5, 50, size= n_rows), 
        "campaign": rng.choice(CAMPAIGNS, size= n_rows), 
        "valid_from": valid_from,
        "valid_to": valid_to
    })




def generate_bookings(customers: pd.DataFrame, rooms: pd.DataFrame, room_types: pd.DataFrame, promotions: pd.DataFrame, n_rows: int) -> pd.DataFrame:

    number_of_nights = rng.integers(1, 14, size= n_rows)
    check_in_date = generate_dates("2022-10-13", "2026-01-01", n_rows)
    check_out_date = check_in_date + pd.to_timedelta(number_of_nights, unit= "D")

    df = pd.DataFrame({
        "booking_id": make_ids("booking", n_rows), 
        "customer_id": rng.choice(customers["customer_id"], size= n_rows), 
        "room_id": rng.choice(rooms["room_id"], size= n_rows), 
        "number_of_nights": number_of_nights, 
        "check_in_date": check_in_date, 
        "check_out_date":check_out_date, 
        "status": rng.choice(["Confirmed", "Cancelled", "Pending"], size= n_rows, p=[0.9, 0.07, 0.03]), 
        "is_full_package": rng.choice(["Yes", "No"], size= n_rows, p= [0.15, 0.85])
    })

    # for promotions
    promo_id = pd.Series([None] * n_rows, dtype= object)
    for p in promotions.itertuples():
        in_window = (p.valid_from <= df["check_in_date"]) & (p.valid_to >= df["check_out_date"])
        take = in_window & promo_id.isna() & (rng.random(n_rows) <= 0.3)
        promo_id[take] = p.promotion_id
    df["promotion_id"] = promo_id

    # total amount = nights * room base cost, +25% full package, minus promo discount
    base = (df[["room_id"]].merge(rooms[["room_id", "room_type_id"]], how= "left", on= "room_id")
                        .merge(room_types[["room_type_id", "base_cost"]], how= "left", on= "room_type_id")
                        ["base_cost"].to_numpy())

    pct = df["promotion_id"].map(promotions.set_index("promotion_id")["discount_pct"]).fillna(0).to_numpy()

    fpkg = np.where(df["is_full_package"] == "Yes", 1.25, 1)

    df["total_amount"] = np.round((base * fpkg * number_of_nights * (1 - pct / 100)), 2)

    return df

df_cust = generate_customers(100)
rooms = generate_rooms(["pr1", "pr2", "pr3", "pr4", "pr5"], ["rtype1", "rtype2", "rtype3", "rtype4"], 100)
room_types = generate_room_types()
df_promo = generate_promotions(30)
# df = generate_proprties(rooms, room_types, ids, 100)

df = generate_bookings(df_cust, rooms, room_types, df_promo, 1000)
df.head()


def generate_payments(bookings: pd.DataFrame, n_rows: int) -> pd.DataFrame:

    paid = bookings[bookings["status"] == "Confirmed"]
    picked = paid.sample(n_rows, replace= True, random_state= SEED).reset_index(drop= True)
    created_at = picked["check_in_date"] - pd.to_timedelta(rng.integers(2, 29), unit= "D")
    
    return pd.DataFrame({
        "payment_id": make_ids("payment", n_rows), 
        "booking_id": picked["booking_id"], 
        "amount": picked["total_amount"], 
        "bank_provider": rng.choice(["mastercard", "visa", "paypal", "stripe", "crypto"], size= n_rows), 
        "status": rng.choice(["Complited", "Rejected"], size= n_rows, p= [0.95, 0.05]), 
        "created_at": created_at
    })
    


def generate_department() -> pd.DataFrame:
    return pd.DataFrame({
    "department_id": make_ids("department", len(DEPARTMENTS)), 
    "name": DEPARTMENTS 
    })



def genearte_employees(properties_ids: list[str], departments: pd.DataFrame, n_rows: int) -> pd.DataFrame:    

    properties_count = len(properties_ids)
    print(properties_count)
    s_span = n_rows - properties_count
    managment_dept = departments[departments["name"] == "Management"]["department_id"].iloc[0]
    other_depts = departments[departments["name"] != "Management"]["department_id"].to_numpy()
    prps = np.concatenate([properties_ids, rng.choice(properties_ids, size= s_span)])
    ids = make_ids('emp', n_rows)
    gm = dict(zip(properties_ids, ids[:properties_count]))
    birthdates = generate_dates("1975-01-10", "2003-02-16", n_rows)
    hire_date = generate_dates(adult_from(birthdates), pd.Timestamp.today().normalize(), n_rows)

    return pd.DataFrame({
        "employee_id": ids, 
        "first_name": [fake.first_name() for _ in range(n_rows)], 
        "last_name": [fake.last_name() for _ in range(n_rows)], 
        "birthdate": birthdates, 
        "hire_date": hire_date, 
        "property_id": prps, 
        "manager_id": [None] * properties_count + [gm[pr] for pr in prps[properties_count:]],
        "department_id": [managment_dept] * properties_count + list(rng.choice(other_depts, size= s_span))
    })





#---------------------------------------------------------------------------------------------------------- generate tables
def generate_tables(n_customers: int, n_properties: int, n_rooms: int, n_promotions: int, n_bookings: int, n_payments: int, n_employees: int) -> dict[str, pd.DataFrame]:
    
    property_ids = make_ids("property", n_properties)
    customers = generate_customers(n_customers)
    room_types_ = generate_room_types()
    rooms = generate_rooms(property_ids, room_types_["room_type_id"].to_list(), n_rooms)
    properties = generate_proprties(rooms, room_types_, property_ids, n_properties)
    promotions = generate_promotions(n_promotions)
    bookings = generate_bookings(customers, rooms, room_types_, promotions, n_bookings)
    payments = generate_payments(bookings, n_payments)
    departments_ = generate_department()
    employees = genearte_employees(property_ids, departments_, n_employees)

    return {
        "customers": customers,
        "room_types": room_types_,
        "rooms": rooms,
        "properties": properties,
        "promotions": promotions,
        "bookings": bookings,
        "payments": payments,
        "departments": departments_,
        "employees": employees
    }


#----------------------------------------------------------------------------------------------------------------------

if __name__ == '__main__':

    tables = generate_tables(2000, 20, 2500, 30, 75000, 80000, 500)
    out = Path("output")
    out.mkdir(exist_ok= True)

    for name, df in tables.items():
        df.to_csv(out / f"{name}.csv", index= False)
        print(name, len(df), 'rows')


20
customers 2000 rows
room_types 4 rows
rooms 2500 rows
properties 20 rows
promotions 30 rows
bookings 75000 rows
payments 80000 rows
departments 7 rows
employees 500 rows


In [15]:
df = pd.read_csv("output\\employees.csv")
df.tail(50)

,employee_id,first_name,last_name,birthdate,hire_date,property_id,manager_id,department_id
450,emp_00451,Rosa,Morell,1987-07-10,2015-06-17,property_00011,emp_00011,department_00004
451,emp_00452,Alexander,Wojtczuk,1978-06-03,2010-10-17,property_00010,emp_00010,department_00007
452,emp_00453,Margaret,Aubert,1976-11-02,1999-02-15,property_00011,emp_00011,department_00006
453,emp_00454,Bruno,Haase,1977-11-05,2021-02-25,property_00005,emp_00005,department_00007
454,emp_00455,Sarah,Kuhl,1999-05-18,2022-09-14,property_00007,emp_00007,department_00003
455,emp_00456,Russell,Howard,1999-09-28,2023-03-25,property_00020,emp_00020,department_00007
456,emp_00457,Sofía,Esteves,1998-06-28,2020-11-23,property_00005,emp_00005,department_00007
457,emp_00458,Lina,Garrido,1982-04-03,2021-07-26,property_00013,emp_00013,department_00007
458,emp_00459,Luca,Anguiano,1996-11-26,2019-02-21,property_00007,emp_00007,department_00006
459,emp_00460,Olgierd,Ochab,1977-03-15,2020-04-05,property_00018,emp_00018,department_00004
